In [1]:
import threading
import time
import logging
import multiprocessing
import os
import shutil
import filecmp
from concurrent.futures import ThreadPoolExecutor, as_completed

Configuarzione del logger

In [2]:
def setup_logging():
    """
    Configuro il logger per tracciare le operazioni.
    Lo istanzio all'interno dei processi per evitare conflitti di lock.
    """
    logger = logging.getLogger('SyncEase')
    if not logger.handlers:
        logger.setLevel(logging.DEBUG)
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - [PID %(process)d] - %(levelname)s - %(message)s')
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)
    return logger

Funzioni operative

In [3]:
def copy_file(task):
    """
    Copia un file dalla sorgente alla destinazione preservando i metadati.
    """
    source_path, destination_path = task
    try:
        os.makedirs(os.path.dirname(destination_path), exist_ok=True)
        shutil.copy2(source_path, destination_path)
        return f"COPIATO: {source_path} -> {destination_path}"
    except Exception as e:
        return f"ERRORE COPIA {source_path}: {e}"


def delete_path(path):
    """
    Elimina file o intere cartelle in modo ricorsivo.
    """
    try:
        if os.path.isfile(path) or os.path.islink(path):
            os.remove(path)
            return f"ELIMINATO FILE: {path}"
        elif os.path.isdir(path):
            shutil.rmtree(path)
            return f"ELIMINATA CARTELLA: {path}"
    except Exception as e:
        return f"ERRORE ELIMINAZIONE {path}: {e}"


def sync_single_directory(args):
    """
    Sincronizzo i file all'interno di una singola directory.
    Utilizzo ThreadPoolExecutor per parallelizzare le operazioni di I/O (copia/cancella).
    """
    source_dir, dest_dir = args
    logger = setup_logging()

    # Assicuro che la cartella di destinazione esista
    if not os.path.exists(dest_dir):
        try:
            os.makedirs(dest_dir, exist_ok=True)
        except FileExistsError:
            pass

    # Confronto il contenuto delle directory
    comparison = filecmp.dircmp(source_dir, dest_dir)

    # 1. Identificazione File da Copiare (Nuovi + Modificati)
    # left_only: file presenti solo in source
    # diff_files: file presenti in entrambi ma diversi
    files_to_process = comparison.left_only + comparison.diff_files

    copy_tasks = []
    for f in files_to_process:
        src = os.path.join(source_dir, f)
        dst = os.path.join(dest_dir, f)

        # Filtro: agisco solo sui file qui.
        if os.path.isfile(src):

            if f in comparison.diff_files:
                src_mtime = os.path.getmtime(src)
                dst_mtime = os.path.getmtime(dst)
                if src_mtime > dst_mtime:
                    copy_tasks.append((src, dst))
            else:

                copy_tasks.append((src, dst))

    # 2. Identificazione File/Cartelle da Eliminare (Obsoleti)
    # right_only: presenti solo in destination
    delete_tasks = []
    for f in comparison.right_only:
        path = os.path.join(dest_dir, f)
        delete_tasks.append(path)

    # 3. Esecuzione Multithreading
    # Max threads hardcoded o calcolati.
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = []


        for task in copy_tasks:
            futures.append(executor.submit(copy_file, task))


        for path in delete_tasks:
            futures.append(executor.submit(delete_path, path))

        # Raccolta risultati
        for future in as_completed(futures):
            res = future.result()
            if "ERRORE" in res:
                logger.error(res)
            else:
                logger.debug(res)


def synchronize_folders(source_folder, destination_folder, max_processes=4):
    """
    Funzione principale.
    Scansiona l'albero delle directory e assegna ogni cartella a un processo dedicato.
    """
    logger = setup_logging()
    start_time = time.time()
    logger.info(f"Avvio sincronizzazione: {source_folder} -> {destination_folder}")


    dirs_to_sync = []


    dirs_to_sync.append((source_folder, destination_folder))

    for root, dirs, _ in os.walk(source_folder):
        for d in dirs:
            src_path = os.path.join(root, d)
            # Calcolo il percorso relativo per replicare la struttura in destinazione
            rel_path = os.path.relpath(src_path, source_folder)
            dst_path = os.path.join(destination_folder, rel_path)
            dirs_to_sync.append((src_path, dst_path))

    logger.info(f"Individuate {len(dirs_to_sync)} directory da analizzare.")

    # Multiprocessing: Ogni processo si occupa di confrontare e sincronizzare una directory specifica
    with multiprocessing.Pool(processes=max_processes) as pool:
        pool.map(sync_single_directory, dirs_to_sync)


    if os.path.exists(destination_folder):
        root_cmp = filecmp.dircmp(source_folder, destination_folder)
        for item in root_cmp.right_only:
            path_to_remove = os.path.join(destination_folder, item)
            delete_path(path_to_remove)
            logger.info(f"Pulizia finale root: Rimosso {path_to_remove}")

    elapsed = time.time() - start_time
    logger.info(f"Sincronizzazione completata in {elapsed:.2f} secondi.")




def create_dummy_files(base_path, folder_name, num_files, content="test"):
    path = os.path.join(base_path, folder_name)
    os.makedirs(path, exist_ok=True)
    for i in range(num_files):
        with open(os.path.join(path, f"file_{i}.txt"), "w") as f:
            f.write(content)
    return path

In [4]:
if __name__ == "__main__":
    # Configuro le directory di test
    BASE_TEST = "test_env"
    SRC = os.path.join(BASE_TEST, "source")
    DST = os.path.join(BASE_TEST, "destination")


    if os.path.exists(BASE_TEST):
        shutil.rmtree(BASE_TEST)
    os.makedirs(SRC)

    print("\n--- CASO 1: Pochi file ---")
    create_dummy_files(SRC, "docs", 5)
    synchronize_folders(SRC, DST)

    print("\n--- CASO 2: Molti file (Stress Test) ---")

    create_dummy_files(SRC, "data", 100)  # 100 file
    synchronize_folders(SRC, DST)

    print("\n--- CASO 3: Sottocartelle e Modifiche ---")
    # 1. Modifico un file esistente
    with open(os.path.join(SRC, "docs", "file_0.txt"), "w") as f:
        f.write("MODIFICATO")
    # 2. Cancello un file nella sorgente (per testare eliminazione in destinazione)
    os.remove(os.path.join(SRC, "docs", "file_1.txt"))
    # 3. Aggiungo deep nesting
    create_dummy_files(os.path.join(SRC, "data"), "2025_archive", 10)

    synchronize_folders(SRC, DST)

    print("\n--- Verifica Finale ---")
    diff = filecmp.dircmp(SRC, DST)
    if not diff.diff_files and not diff.left_only and not diff.right_only:
        print("SUCCESSO: Directory perfettamente sincronizzate.")
    else:
        print("ATTENZIONE: Trovate differenze!")
        diff.report()

2025-12-24 10:22:12,284 - [PID 491] - INFO - Avvio sincronizzazione: test_env/source -> test_env/destination
INFO:SyncEase:Avvio sincronizzazione: test_env/source -> test_env/destination
2025-12-24 10:22:12,286 - [PID 491] - INFO - Individuate 2 directory da analizzare.
INFO:SyncEase:Individuate 2 directory da analizzare.



--- CASO 1: Pochi file ---


DEBUG:SyncEase:COPIATO: test_env/source/docs/file_2.txt -> test_env/destination/docs/file_2.txt
DEBUG:SyncEase:COPIATO: test_env/source/docs/file_1.txt -> test_env/destination/docs/file_1.txt
DEBUG:SyncEase:COPIATO: test_env/source/docs/file_4.txt -> test_env/destination/docs/file_4.txt
DEBUG:SyncEase:COPIATO: test_env/source/docs/file_3.txt -> test_env/destination/docs/file_3.txt
DEBUG:SyncEase:COPIATO: test_env/source/docs/file_0.txt -> test_env/destination/docs/file_0.txt
2025-12-24 10:22:12,374 - [PID 491] - INFO - Sincronizzazione completata in 0.09 secondi.
INFO:SyncEase:Sincronizzazione completata in 0.09 secondi.
2025-12-24 10:22:12,382 - [PID 491] - INFO - Avvio sincronizzazione: test_env/source -> test_env/destination
INFO:SyncEase:Avvio sincronizzazione: test_env/source -> test_env/destination
2025-12-24 10:22:12,384 - [PID 491] - INFO - Individuate 3 directory da analizzare.
INFO:SyncEase:Individuate 3 directory da analizzare.



--- CASO 2: Molti file (Stress Test) ---


DEBUG:SyncEase:COPIATO: test_env/source/data/file_10.txt -> test_env/destination/data/file_10.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_0.txt -> test_env/destination/data/file_0.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_11.txt -> test_env/destination/data/file_11.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_14.txt -> test_env/destination/data/file_14.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_19.txt -> test_env/destination/data/file_19.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_2.txt -> test_env/destination/data/file_2.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_20.txt -> test_env/destination/data/file_20.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_21.txt -> test_env/destination/data/file_21.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_22.txt -> test_env/destination/data/file_22.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/file_23.txt -> test_env/destination/data/file_23.txt
DEBUG:SyncEase:COPIATO: 


--- CASO 3: Sottocartelle e Modifiche ---


DEBUG:SyncEase:ELIMINATO FILE: test_env/destination/docs/file_1.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/2025_archive/file_1.txt -> test_env/destination/data/2025_archive/file_1.txt
DEBUG:SyncEase:COPIATO: test_env/source/docs/file_0.txt -> test_env/destination/docs/file_0.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/2025_archive/file_2.txt -> test_env/destination/data/2025_archive/file_2.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/2025_archive/file_0.txt -> test_env/destination/data/2025_archive/file_0.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/2025_archive/file_4.txt -> test_env/destination/data/2025_archive/file_4.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/2025_archive/file_3.txt -> test_env/destination/data/2025_archive/file_3.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/2025_archive/file_6.txt -> test_env/destination/data/2025_archive/file_6.txt
DEBUG:SyncEase:COPIATO: test_env/source/data/2025_archive/file_7.txt -> test_env/destination/data/2025_a


--- Verifica Finale ---
SUCCESSO: Directory perfettamente sincronizzate.
